# Lab 02: Webhook Integration — Fundamentals

**Duration**: ~10 minutes  
**Prerequisites**: Completed Lab 01 notebooks

## Learning Objectives

By the end of this notebook, you will:
- Understand what webhooks are and why polling is inferior
- Know the most common Stripe event types
- Understand webhook event structure
- Understand why and how signature verification works

---

## What Are Webhooks?

**Webhooks** are HTTP callbacks that notify your application when events happen in Stripe.

### Polling (Pull) vs Webhooks (Push)

| Polling | Webhooks |
|---------|----------|
| Your app asks "any updates?" on a timer | Stripe tells you the moment something happens |
| Wastes resources checking constantly | Efficient — only notified when needed |
| May miss events between polls | Guaranteed delivery with retries |
| Adds latency | Near real-time |

## How Webhooks Work

```
1. An event occurs in Stripe (e.g., customer pays)
         |
         ▼
2. Stripe sends HTTP POST to your endpoint
   Stripe API  ──── POST /webhook ────►  Your Server
   Stripe API  ◄─── 200 OK ───────────  Your Server
         |
         ▼
3. Your app processes the event
   - Verify signature
   - Update database
   - Send confirmation email
   - Fulfill order
```

---

## Common Event Types

Stripe has [hundreds of event types](https://stripe.com/docs/api/events/types). These are the ones you'll use most:

### Payment Events

| Event | When it fires |
|-------|---------------|
| `payment_intent.succeeded` | Payment completed successfully |
| `payment_intent.payment_failed` | Payment attempt failed |
| `charge.refunded` | A charge was refunded |
| `charge.dispute.created` | Customer disputed a charge |

### Subscription Events

| Event | When it fires |
|-------|---------------|
| `customer.subscription.created` | New subscription started |
| `customer.subscription.updated` | Plan, quantity, or status changed |
| `customer.subscription.deleted` | Subscription canceled |
| `invoice.paid` | Invoice payment succeeded |
| `invoice.payment_failed` | Invoice payment failed |

### Customer Events

| Event | When it fires |
|-------|---------------|
| `customer.created` | New customer object created |
| `payment_method.attached` | Payment method added to customer |

---

## Webhook Event Structure

Every webhook event has the same envelope:

```json
{
  "id": "evt_1234567890",
  "object": "event",
  "type": "payment_intent.succeeded",
  "created": 1234567890,
  "data": {
    "object": {
      "id": "pi_1234567890",
      "amount": 2000,
      "currency": "usd"
    }
  }
}
```

| Field | Description |
|-------|-------------|
| `id` | Unique event ID — use this for idempotency |
| `type` | The event type (e.g., `payment_intent.succeeded`) |
| `data.object` | The full Stripe object that triggered the event |
| `created` | Unix timestamp of when the event occurred |

Let's retrieve a real event from the Stripe API to inspect its structure.

In [1]:
!pip install stripe --quiet

import stripe
import json

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    print(f"Connected: {account.id}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.7 MB/s eta 0:00:00
Connected: acct_1RnL4mBMxfUzotEq


In [2]:
# Create a payment so there's a recent event to look at
pi = stripe.PaymentIntent.create(
    amount=1500,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

print(f"Created PaymentIntent: {pi.id} | status: {pi.status}")

Created PaymentIntent: pi_3TMUhmBMxfUzotEq04rZlOw6 | status: succeeded


In [4]:
import time
time.sleep(2)  # Wait a moment for the event to be available

# Retrieve the corresponding event from the Stripe API
events = stripe.Event.list(
    type="payment_intent.succeeded",
    limit=1
)

if events.data:
    event = events.data[0]
    print(f"Event ID:      {event.id}")
    print(f"Event type:    {event.type}")
    print(f"Created:       {event.created}")
    # Fix: Convert event.data.object to a dictionary before getting keys
    print(f"\ndata.object keys: {list(event.data.object.to_dict().keys())}")

    # Access the PaymentIntent inside the event
    payment_intent_from_event = event.data.object
    print(f"\nPaymentIntent ID:     {payment_intent_from_event.id}")
    print(f"Amount:               ${payment_intent_from_event.amount / 100:.2f}")
    print(f"Status:               {payment_intent_from_event.status}")
else:
    print("No events found yet. Try running the previous cell again.")

Event ID:      evt_3TMUhmBMxfUzotEq066katdP
Event type:    payment_intent.succeeded
Created:       1776264374

data.object keys: ['id', 'object', 'amount', 'amount_capturable', 'amount_details', 'amount_received', 'application', 'application_fee_amount', 'automatic_payment_methods', 'canceled_at', 'cancellation_reason', 'capture_method', 'client_secret', 'confirmation_method', 'created', 'currency', 'customer', 'customer_account', 'description', 'excluded_payment_method_types', 'last_payment_error', 'latest_charge', 'livemode', 'managed_payments', 'metadata', 'next_action', 'on_behalf_of', 'payment_method', 'payment_method_configuration_details', 'payment_method_options', 'payment_method_types', 'processing', 'receipt_email', 'review', 'setup_future_usage', 'shipping', 'source', 'statement_descriptor', 'statement_descriptor_suffix', 'status', 'transfer_data', 'transfer_group']

PaymentIntent ID:     pi_3TMUhmBMxfUzotEq04rZlOw6
Amount:               $15.00
Status:               succeede

---

## Signature Verification — Critical for Security

Your webhook endpoint is a public URL. Without verification, anyone could send fake events to it.

### How It Works

1. Stripe signs each webhook payload with an **endpoint-specific secret** (`whsec_...`)
2. The signature is sent in the `Stripe-Signature` HTTP header
3. Your server calls `stripe.Webhook.construct_event()` to verify it

```python
import stripe

endpoint_secret = 'whsec_...'  # From Dashboard or stripe listen

@app.route('/webhook', methods=['POST'])
def webhook():
    payload    = request.data          # Raw bytes — must NOT be parsed first
    sig_header = request.headers.get('Stripe-Signature')

    try:
        event = stripe.Webhook.construct_event(payload, sig_header, endpoint_secret)
    except stripe.error.SignatureVerificationError:
        return 'Invalid signature', 400

    # Safe to process the event
    ...
```

### What Happens Without Verification?

An attacker could:
- Send a fake `payment_intent.succeeded` event to unlock paid features
- Replay old events to trigger duplicate actions
- Cause your backend to take unintended actions

---

## Best Practices

### 1. Always Verify Signatures
Never skip signature verification in production.

### 2. Respond Quickly (< 30 seconds)
Stripe considers your endpoint timed out if it doesn't respond in 30 seconds. For slow operations:
- Return `200 OK` immediately
- Process the event asynchronously (queue, background job)

### 3. Handle Idempotency
Stripe may deliver the same event more than once. Use `event.id` to deduplicate:

```python
if already_processed(event['id']):
    return 200  # Already handled, acknowledge and skip
```

### 4. Handle the `else` Case
Always include a fallthrough for event types your handler doesn't recognise. Stripe adds new event types over time.

```python
if event['type'] == 'payment_intent.succeeded':
    ...
else:
    print(f"Unhandled event type: {event['type']}")

return 200  # Always acknowledge
```

### 5. Log Everything
Log all received events (including unhandled ones) for debugging and audit purposes.

---

## Summary

- **Webhooks** push notifications from Stripe to your server — no polling needed
- Every event has the same structure: `id`, `type`, `data.object`
- **Signature verification** (`construct_event`) is non-negotiable in production
- Return `200 OK` quickly and process asynchronously for slow operations
- Use `event.id` to make your handler idempotent

## Next Steps

Open `05_webhook_server.ipynb` to build a working webhook handler and test it end-to-end.